TEAM 13 \
IOANNIDIS GEORGIOS 10833\
PAPADOPOULOS CHARALAMPOS PANAGIOTIS 10898

**PART D (The final experiment that gives the labels is 5. Ensemble Learning, which does Majority Voting over the labels of experiments 4.4.1, 4.4.2, 4.4.3)**

**1.MLP**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_predict
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from google.colab import files

# --- STEP 1: Load & prepare the data ---
print("⏳ Loading data...")

if os.path.exists('datasetTV.csv') and os.path.exists('datasetTest.csv'):
    train_df = pd.read_csv('datasetTV.csv', header=None)
    test_df = pd.read_csv('datasetTest.csv', header=None)
else:
    raise SystemExit("Error: the files are missing.")

X = train_df.iloc[:, :-1].values
y = train_df.iloc[:, -1].values

# Encode the labels (1-5 to 0-4 so it works properly)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split (Train / Validation)
# Keep the split so we have X_train (for the grid) and X_val for checking
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

print(f" Data ready: Train shape {X_train.shape}")


# --- STEP 2: Pipeline & Grid Search setup ---
# The Pipeline scales each fold on its own, so there is no data leakage.
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(max_iter=500, early_stopping=True, random_state=42,solver='adam'))
])

# --- STEP 3: The Grid Search  ---
param_grid = {
    # Architectures
    'mlp__hidden_layer_sizes': [
        (256, 128),
        (512, 256, 128),
        (128, 64)
    ],
    # Activation
    'mlp__activation': ['relu', 'tanh'],

    # Regularization (Weight Decay)
    'mlp__alpha': [0.0001, 0.001, 0.01],

    #Learning Rate
    'mlp__learning_rate_init': [0.001, 0.01]
}

print("\n Starting Grid Search with Pipeline (3-Fold CV)...")
print("(Same parameters as the 1st experiment, but without data leakage)")

grid_search = GridSearchCV(
    estimator=pipe,       # Pass the pipe
    param_grid=param_grid,
    cv=3,                 # 3-fold Cross Validation
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

# Training
grid_search.fit(X_train, y_train)


print(f"\nBest Params: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

# Keep the best model
best_pipe = grid_search.best_estimator_

# --- STEP 4: Evaluation on the validation set (20% hold-out) ---

print("\n" + "="*60)
print(" Evaluation on the Validation Set (Unseen Data)")
print("="*60)

# 1. Predict on X_val
# The pipeline also scales the validation set automatically
y_val_pred = best_pipe.predict(X_val)

# 2. Decode the labels (0,1,2... back to 1,2,3...)
y_val_true_decoded = le.inverse_transform(y_val)
y_val_pred_decoded = le.inverse_transform(y_val_pred)

# 3. Print the classification report
print("\n--- Validation Classification Report ---")
print(classification_report(y_val_true_decoded, y_val_pred_decoded))

# 4. Show the confusion matrix
plt.figure(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    y_val_true_decoded, y_val_pred_decoded,
    normalize='true',
    values_format='.2%',
    cmap='Blues'
)
plt.title("Confusion Matrix - Validation Set\n(Best Model from Grid Search)")
plt.show()

#show val accuracy separately
val_accuracy = best_pipe.score(X_val, y_val)
print(f"\n Final accuracy on the Validation Set: {val_accuracy * 100:.2f}%")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Get and sort the results
results = pd.DataFrame(grid_search.cv_results_)
# Take the top 15
top_results = results.sort_values(by='rank_test_score').head(15).copy()

# 2. Function that SHORTENS the labels
def get_short_label(params):
    # Convert to string
    p = str(params)
    # Extract the values
    act = params.get('mlp__activation', '?')
    layers = str(params.get('mlp__hidden_layer_sizes', '?')).replace(" ", "") # Remove spaces
    alpha = params.get('mlp__alpha', '?')
    lr = params.get('mlp__learning_rate_init', '?')

    # Build a short string
    # L = Layers, a = alpha, lr = learning rate
    return f"{act} | L:{layers} | a:{alpha} | lr:{lr}"

# Apply the function
top_results['short_label'] = top_results['params'].apply(get_short_label)

# 3. Create the plot
plt.figure(figsize=(13, 8))

# Colour palette
norm = plt.Normalize(top_results['mean_test_score'].min(), top_results['mean_test_score'].max())
colors = plt.cm.viridis(norm(top_results['mean_test_score']))

# Horizontal Bar Plot
bars = plt.barh(top_results['short_label'],
                top_results['mean_test_score'] * 100,
                xerr=top_results['std_test_score'] * 100, # Error bars
                color=colors, capsize=4)

# 4. Formatting
plt.xlabel('Validation Accuracy (%)', fontsize=12, fontweight='bold')
plt.ylabel('Configurations (Act | Layers | Alpha | LR)', fontsize=12, fontweight='bold')
plt.title('Top 15 MLP Combinations', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis() # Best one at the top
plt.grid(axis='x', linestyle='--', alpha=0.5)


min_score = (top_results['mean_test_score'].min() * 100) - 0.5
max_score = (top_results['mean_test_score'].max() * 100) + 0.5
plt.xlim(min_score, max_score)

# Add the values on the bars
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.05, bar.get_y() + bar.get_height()/2,
             f'{width:.2f}%', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

**2.RANOM FOREST**

In [ ]:
#Random Forest
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from google.colab import files
import io
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import cuml
from cuml.ensemble import RandomForestClassifier as cuRF
from sklearn.model_selection import GridSearchCV, train_test_split
import seaborn as sns
import sklearn

#Upload files
print("Please upload the files datasetTV.csv and datasetTest.csv:")
uploaded = files.upload()

# Check they were uploaded correctly
try:
    train_df = pd.read_csv('datasetTV.csv', header=None)
    test_df = pd.read_csv('datasetTest.csv', header=None)
    print("Files loaded successfully")
except FileNotFoundError:
    print("Error: files not found")

#Prepare the data
# Convert to float32 to speed up the GPU
X = train_df.values[:, :-1].astype(np.float32)
y = train_df.values[:, -1].astype(np.float32)
X_test=test_df.values.astype(np.float32)

#Train/Validation 80/20 Split
X_train_internal, X_val_internal, y_train_internal, y_val_internal = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# cuML model
cuml_model = cuRF(random_state=42)

# Parameters to try in the grid search
param_grid = {
    'n_estimators': [100, 300, 500],# How many trees vote

    'max_depth': [10, 20, 30, None],# How deep the trees can go

    'max_features': ['sqrt', 1.0],# How many features to look at in each split.

    'min_samples_leaf': [1, 2, 4],# Minimum samples per leaf

    'bootstrap': [True, False]# Whether to reuse samples
}

#
grid_search = GridSearchCV(
    estimator=cuml_model,
    param_grid=param_grid,
    cv=3,# We use cv=3 (3-fold Cross Validation)
    verbose=1,
    scoring='accuracy'# Our goal is maximum accuracy.
)

#grid search
grid_search.fit(X_train_internal, y_train_internal)

# Results
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best accuracy (Cross-Validation): {grid_search.best_score_ * 100:.2f}%")

# 1. Prepare the data from the grid search
results = pd.DataFrame(grid_search.cv_results_)

# Convert the parameters to strings
param_cols = [c for c in results.columns if c.startswith('param_')]
for col in param_cols:
    results[col] = results[col].astype(str)

# --- SEPARATE LINE PLOTS ---

print("\n📊 Trend plot: mean accuracy per parameter")

for col in param_cols:

    clean_name = col.replace("param_mlp__", "")

    # GROUPING: mean score for each value of the parameter
    agg_data = results.groupby(col)['mean_test_score'].mean() * 100

    try:
        # If they are numbers, sort numerically
        sorted_index = sorted(agg_data.index, key=lambda x: float(x))
        agg_data = agg_data.reindex(sorted_index)
    except:
        # If they are strings, sort alphabetically
        agg_data = agg_data.sort_index()

    # Plot
    plt.figure(figsize=(10, 6))

    # Line Plot
    plt.plot(agg_data.index, agg_data.values, marker='o', linestyle='-', color='#1f77b4', linewidth=2.5, markersize=8)

    # Add labels (the % accuracy) on each point
    for x, y in zip(agg_data.index, agg_data.values):
        plt.text(x, y + 0.05, f'{y:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)

    # Formatting
    plt.title(f'Effect of parameter: {clean_name}\n(Mean Parameter vs Accuracy)', fontsize=14, fontweight='bold')
    plt.xlabel(clean_name, fontsize=12)
    plt.ylabel('Mean Accuracy (%)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)

    # Set the Y limits
    plt.ylim(agg_data.min() - 0.5, agg_data.max() + 0.5)

    plt.tight_layout()
    plt.show()

#Final check on the hold-out set (20%)
best_model = grid_search.best_estimator_
val_predictions = best_model.predict(X_val_internal)
final_metrics=sklearn.metrics.classification_report(y_val_internal, val_predictions,digits=4)
print(final_metrics)


# Build the confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_val_internal,
    val_predictions,
    normalize='true',
    values_format='.2%',
    cmap='Blues',
    display_labels=[1, 2, 3, 4, 5]
)

plt.title('Confusion Matrix (Percentages)')
plt.show()

**3.XGBoost**

In [ ]:
#XGBoost
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Relabelling
# XGBoost needs the classes to start from 0 ( 0, 1, 2, 3, 4)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_internal)
y_val_enc = le.transform(y_val_internal)

#Parameters for the grid search
param_dist = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2], #Learning rate
    'max_depth': [4, 6, 8, 10],#Tree depth
    'min_child_weight': [1, 3, 5],# Leaf weight
    'subsample': [0.7, 0.8, 0.9],# Fraction of samples
    'colsample_bytree': [0.7, 0.8, 0.9],# Fraction of features
    'n_estimators': [500, 1000],# Number of trees
    'gamma': [0, 0.1, 0.5] # Split threshold
}

#Define the base model
xgb_base = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    tree_method='hist',
    device='cuda',        #ENABLE GPU
    random_state=42
)

# --- STEP 5c: Run the search (RandomizedSearchCV) ---
print("Starting Grid Search on GPU...")

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=15,            # Tries 15 random combinations
    scoring='accuracy',   # Evaluation metric
    cv=3,                 # 3-fold Cross Validation
    verbose=1,
    random_state=42
)

# Training
random_search.fit(X_train_internal, y_train_enc)

print("\n📊 Summary of results (sorted):")

# Convert to DataFrame and show
results_df = pd.DataFrame(random_search.cv_results_)
view = results_df[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']]
print(view.sort_values(by='rank_test_score').head(10))

#Evaluate the best model (Best Estimator)
print(f"\nBest Params found: {random_search.best_params_}")

best_xgb = random_search.best_estimator_

# Predictions on the validation set
preds_enc = best_xgb.predict(X_val_internal)
y_pred_val = le.inverse_transform(preds_enc)

print("\n--- Optimized XGBoost Classification Report (Validation) ---")
print(classification_report(y_val_internal, y_pred_val))

# 1. Get the predictions of the best XGBoost model
preds_enc = best_xgb.predict(X_val_internal)

# 2. Map the predictions back to the real labels (1, 2, 3, 4, 5)
val_predictions = le.inverse_transform(preds_enc)

# 3. Create the plot
plt.figure(figsize=(10, 8))
# Compare y_val_internal (true) with val_predictions (predicted)
# Build the confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_val_internal,
    val_predictions,
    normalize='true',
    values_format='.2%',
    cmap='Blues',
    display_labels=le.classes_
)

plt.title('XGBoost Confusion Matrix (Validation Set)')
plt.grid(False)
plt.show()

**4.SVM**

**4.1 Different Kernels**

In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import cuml
from cuml.svm import SVC as GPU_SVC
print(f" GPU Mode Active: cuML version {cuml.__version__}")

# =============================================================================
# 1. LOAD THE DATA
# =============================================================================
print("\n--- 1. Loading data ---")
# Check that the files exist
if os.path.exists('datasetTV.csv') and os.path.exists('datasetTest.csv'):
    train_df = pd.read_csv('datasetTV.csv', header=None)
    test_df = pd.read_csv('datasetTest.csv', header=None)
    print(" Files found and loaded!")
else:
    raise SystemExit("Error: the files are missing")

# Convert to float32 for better GPU performance
trainData = train_df.values.astype('float32')
testData = test_df.values.astype('float32')

#split into features and labels
X = trainData[:, :-1]
y = trainData[:, -1]

# 80-20 split (the 20 is the validation hold-out)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# COMPUTE THE SCALE VALUE (1 / (n_features * X.var()))
n_features = X_train.shape[1]
gamma_scale_approx = 1.0 / n_features
print(f"Training Set: {X_train.shape}")
print(f"  The 'scale' value is: {gamma_scale_approx:.6f}")

# =============================================================================
# 2. PIPELINE SETUP & GRID SEARCH
# =============================================================================
print("\n--- 2. Starting Grid Search (RBF, Poly, Sigmoid) ---")

#use a pipeline to avoid data leakage
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', GPU_SVC(
        class_weight='balanced',#balances the different number of samples per class
        cache_size=2000,
        probability=False
    ))
])

# WE TRY 3 DIFFERENT KERNELS
param_grid = [
    # 1. RBF Kernel
    {
        'svm__kernel': ['rbf'],
        'svm__C': [1, 10, 50],
        'svm__gamma': ['scale', 0.01, 0.005]
    },

    # 2. Polynomial Kernel
    {
        'svm__kernel': ['poly'],
        'svm__C': [1, 10],
        'svm__degree': [2, 3],
        'svm__gamma': ['scale'],
        'svm__coef0': [0.0, 1.0]
    },

    # 3. Sigmoid Kernel
    {
        'svm__kernel': ['sigmoid'],
        'svm__C': [1, 10],
        'svm__gamma': ['scale', 0.01],
        'svm__coef0': [0.0, 1.0]
    }
]

# Run GridSearch with 5-Fold Cross Validation.
grid_search = GridSearchCV(
    pipe, #use the pipeline we built
    param_grid,
    refit=True,  #the model is retrained on all the data (80%)
    verbose=2,
    cv=5,
    n_jobs=1,
    return_train_score=True
)

start_time = time.time()
grid_search.fit(X_train, y_train)
end_time = time.time()
print(f" Finished in {end_time - start_time:.2f} seconds.")

# =============================================================================
# 3. ANALYSIS AND VISUALIZATION OF THE RESULTS
# =============================================================================
results_df = pd.DataFrame(grid_search.cv_results_)

# Get the parameters to make readable labels
results_df['param_kernel'] = results_df['param_svm__kernel']
results_df['param_C'] = results_df['param_svm__C']
results_df['param_gamma'] = results_df['param_svm__gamma']
results_df['param_degree'] = results_df.get('param_svm__degree', np.nan)
results_df['param_coef0'] = results_df.get('param_svm__coef0', np.nan)

# Labeling function so the parameters show in every case
def get_full_label(row):
    k = row['param_kernel']
    c = row['param_C']
    g = row['param_gamma']

    # Convert gamma if it is 'scale'
    g_str = f"scale ({gamma_scale_approx:.4f})" if g == 'scale' else f"{g}"

    base = f"{k.upper()} | C:{c}, G:{g_str}"

    if k == 'poly':
        d = row['param_degree']
        coef = row['param_coef0']
        return f"{base}, Deg:{d}, Coef:{coef}"
    elif k == 'sigmoid':
        coef = row['param_coef0']
        return f"{base}, Coef:{coef}"
    else: # RBF
        return base

results_df['params_str'] = results_df.apply(get_full_label, axis=1)

# Sort the results by validation performance
results_df_sorted = results_df.sort_values(by='mean_test_score', ascending=False).reset_index(drop=True)


print("\n" + "="*90)
print("SORTED RESULTS (ALL KERNELS)")
print("-" * 90)
print(f"{'Parameters':<60} | {'Train':<8} | {'Val(CV)':<8}")
print("-" * 90)
# Print a table to compare train vs validation score (overfitting check)
for index, row in results_df_sorted.iterrows():
    train_acc = row.get('mean_train_score', 0) * 100
    val_acc = row['mean_test_score'] * 100
    print(f"{row['params_str']:<60} | {train_acc:.2f}%   | {val_acc:.2f}%")
print("="*90 + "\n")

# --- PLOTS IN GROUPS OF 8 SO THEY STAY READABLE ---
BATCH_SIZE = 8
total_experiments = len(results_df_sorted)

for i in range(0, total_experiments, BATCH_SIZE):
    subset_df = results_df_sorted.iloc[i : i + BATCH_SIZE]
    # Prepare the data for seaborn
    acc_data_subset = subset_df[['params_str', 'mean_train_score', 'mean_test_score']].melt(
        id_vars='params_str', var_name='Type', value_name='Accuracy'
    )
    acc_data_subset['Type'] = acc_data_subset['Type'].replace({
        'mean_train_score': 'Train Accuracy', 'mean_test_score': 'Validation Accuracy'
    })

    fig, ax = plt.subplots(1, 2, figsize=(22, 9))
    plt.suptitle(f"Results: experiments {i+1} to {min(i+BATCH_SIZE, total_experiments)}", fontsize=16)

    # Plot Accuracy
    sns.barplot(x='params_str', y='Accuracy', hue='Type', data=acc_data_subset, ax=ax[0], palette='viridis')
    ax[0].set_title('Accuracy Comparison', fontsize=15, fontweight='bold')
    ax[0].set_ylim(0.50, 1.01)
    ax[0].grid(axis='y', linestyle='--', alpha=0.5)
    ax[0].set_xticklabels(ax[0].get_xticklabels(), rotation=45, ha='right', fontsize=9)

    # Plot Time
    sns.barplot(x='params_str', y='mean_fit_time', data=subset_df, ax=ax[1], palette='Reds_r')
    ax[1].set_title('Training Time (Sec)', fontsize=15, fontweight='bold')
    ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=45, ha='right', fontsize=9)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# =============================================================================
# 4.EVALUATION OF THE BEST MODEL
# =============================================================================
best_model = grid_search.best_estimator_
y_pred_val = best_model.predict(X_val)

if hasattr(y_pred_val, 'get'): y_pred_val = y_pred_val.get()
if hasattr(y_pred_val, 'to_numpy'): y_pred_val = y_pred_val.to_numpy()

print(f"\n Best Params: {grid_search.best_params_}")
print("--- Classification Report (Validation Set) ---")
print(classification_report(y_val, y_pred_val, digits=4))

# Confusion matrix to show the classification errors
cm = confusion_matrix(y_val, y_pred_val)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix (Kernel: {grid_search.best_params_["svm__kernel"]})')
plt.show()


**4.2 RBF Kernel**

In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import cuml
from cuml.svm import SVC as GPU_SVC
print(f" GPU Mode Active: cuML version {cuml.__version__}")

# =============================================================================
# 1. LOAD THE DATA
# =============================================================================
print("\n--- 1. Loading data ---")

if os.path.exists('datasetTV.csv') and os.path.exists('datasetTest.csv'):
    train_df = pd.read_csv('datasetTV.csv', header=None)
    test_df = pd.read_csv('datasetTest.csv', header=None)
    print(" Files found and loaded!")
else:
    raise SystemExit(" Error: the files are missing.")

# Convert to float32 for better GPU performance
trainData = train_df.values.astype('float32')
testData = test_df.values.astype('float32')

#split into features and labels
X = trainData[:, :-1]
y = trainData[:, -1]

# 80-20 split (the 20 is the validation hold-out)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# COMPUTE THE SCALE VALUE
n_features = X_train.shape[1]
gamma_scale_approx = 1.0 / n_features
print(f"Training Set: {X_train.shape}")
print(f" The 'scale' value is: {gamma_scale_approx:.6f}")

# =============================================================================
# 2. PIPELINE SETUP & GRID SEARCH
# =============================================================================
print("\n--- 2. Starting Grid Search (GPU Mode) ---")

#use a pipeline to avoid data leakage
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', GPU_SVC(
        kernel='rbf',
        class_weight='balanced',#balances the different number of samples per class
        cache_size=2000,
        probability=False
    ))
])

#For the other experiments (fine searches) with RBF we just changed the hyperparameter values
param_grid = {   # RBF Kernel
    'svm__C': [1, 5,10, 50],
    'svm__gamma': ['scale', 0.01, 0.001,0.0001]
}

# Run GridSearch with 5-Fold Cross Validation.
grid_search = GridSearchCV(
    pipe,
    param_grid,
    refit=True, #the model is retrained on all the data (80%)
    verbose=2,
    cv=5,
    n_jobs=1,
    return_train_score=True
)

start_time = time.time()
grid_search.fit(X_train, y_train)
end_time = time.time()
print(f"Finished in {end_time - start_time:.2f} seconds.")

# =============================================================================
# 3.  ANALYSIS AND VISUALIZATION OF THE RESULTS
# =============================================================================
results_df = pd.DataFrame(grid_search.cv_results_)

# Get the parameters to make readable labels
results_df['param_C'] = results_df['param_svm__C']
results_df['param_gamma'] = results_df['param_svm__gamma']

# Labeling function so the parameters show in every case
def get_full_label(row):
    c = row['param_C']
    g = row['param_gamma']

    if g == 'scale':
        return f"C:{c}, G:scale ({gamma_scale_approx:.5f})"
    else:
        return f"C:{c}, G:{g}"

results_df['params_str'] = results_df.apply(get_full_label, axis=1)

# Sort the results by validation performance
results_df_sorted = results_df.sort_values(by='mean_test_score', ascending=False).reset_index(drop=True)


print("\n" + "="*90)
print("SORTED RESULTS (RBF Kernel)")
print("-" * 90)
print(f"{'Parameters':<45} | {'Train Acc':<12} | {'Val Acc (CV)':<12}")
print("-" * 90)
# Print a table to compare train vs validation score (overfitting check)
for index, row in results_df_sorted.iterrows():
    train_acc = row.get('mean_train_score', 0) * 100
    val_acc = row['mean_test_score'] * 100
    print(f"{row['params_str']:<45} | {train_acc:.2f}%      | {val_acc:.2f}%")
print("="*90 + "\n")

# --- PLOTS IN GROUPS OF 8 SO THEY STAY READABLE ---
BATCH_SIZE = 8
total_experiments = len(results_df_sorted)

for i in range(0, total_experiments, BATCH_SIZE):
    subset_df = results_df_sorted.iloc[i : i + BATCH_SIZE]

    acc_data_subset = subset_df[['params_str', 'mean_train_score', 'mean_test_score']].melt(
        id_vars='params_str', var_name='Type', value_name='Accuracy'
    )
    acc_data_subset['Type'] = acc_data_subset['Type'].replace({
        'mean_train_score': 'Train Accuracy', 'mean_test_score': 'Validation Accuracy'
    })

    fig, ax = plt.subplots(1, 2, figsize=(22, 9))
    plt.suptitle(f"Results: experiments {i+1} to {min(i+BATCH_SIZE, total_experiments)} (Sorted)", fontsize=16)

    # Plot Accuracy
    sns.barplot(x='params_str', y='Accuracy', hue='Type', data=acc_data_subset, ax=ax[0], palette='viridis')
    ax[0].set_title('Accuracy Comparison (Overfitting Check)', fontsize=15, fontweight='bold')
    ax[0].set_ylim(0.80, 1.01)
    ax[0].grid(axis='y', linestyle='--', alpha=0.5)
    ax[0].set_xlabel('Parameters (C, Gamma)', fontsize=13)
    ax[0].set_xticklabels(ax[0].get_xticklabels(), rotation=45, ha='right', fontsize=10)

    # Plot Time
    sns.barplot(x='params_str', y='mean_fit_time', data=subset_df, ax=ax[1], palette='Reds_r')
    ax[1].set_title('Training Time (Seconds)', fontsize=15, fontweight='bold')
    ax[1].set_ylabel('Seconds')
    ax[1].set_xlabel('Parameters (C, Gamma)', fontsize=13)
    ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=45, ha='right', fontsize=10)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# =============================================================================
# 4. EVALUATION OF THE BEST MODEL
# =============================================================================
best_model = grid_search.best_estimator_
y_pred_val = best_model.predict(X_val)

# Move from GPU array to CPU if needed
if hasattr(y_pred_val, 'get'): y_pred_val = y_pred_val.get()
if hasattr(y_pred_val, 'to_numpy'): y_pred_val = y_pred_val.to_numpy()

print(f"\n Best Params: {grid_search.best_params_}")
if grid_search.best_params_['svm__gamma'] == 'scale':
    print(f"   (scale value ≈ {gamma_scale_approx:.6f})")

print("--- Classification Report (Validation Set) ---")
print(classification_report(y_val, y_pred_val, digits=4))

cm = confusion_matrix(y_val, y_pred_val)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix (Validation Set)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()


**4.2.1 Evaluation of the best models (MEAN Acc and SVs)**

In [ ]:
import pandas as pd
import numpy as np
import time
import os

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import cuml
from cuml.svm import SVC as GPU_SVC
print(f" GPU Mode Active: cuML version {cuml.__version__}")

# =============================================================================
# 1. LOAD THE DATA
# =============================================================================
print("\n--- 1. Loading data ---")

# Check that the files exist
if os.path.exists('datasetTV.csv') and os.path.exists('datasetTest.csv'):
    train_df = pd.read_csv('datasetTV.csv', header=None)
    test_df = pd.read_csv('datasetTest.csv', header=None)
    print(" Files found and loaded!")
else:
    raise SystemExit(" Error: the files are missing.")

# Convert to float32 for better GPU performance
trainData = train_df.values.astype('float32')
testData = test_df.values.astype('float32')

#split into features and labels
X = trainData[:, :-1]
y = trainData[:, -1]

# Set up 5-Fold Cross Validation
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# =============================================================================
# 2. EVALUATION (5-FOLD CV) & TRAINING OF 4 MODELS
# =============================================================================
print("\n" + "="*90)
print("5-FOLD CROSS VALIDATION ANALYSIS FOR 4 MODELS")
print("="*90)

models_to_test = [
    {'C': 2.0, 'gamma': 0.005, 'name': 'Model_1_(C2_G0.005)'},
    {'C': 1.0, 'gamma': 'scale', 'name': 'Model_2_(C1_Gscale)'},
    {'C': 3.0, 'gamma': 'scale', 'name': 'Model_3_(C3_Gscale)'},
    {'C': 2.72, 'gamma': 0.0058, 'name': 'Model_4_(C2.72_G0.0058)'}
]

for params in models_to_test:
    c_val = params['C']
    g_val = params['gamma']
    name = params['name']

    print(f"\n MODEL: {name}")
    print(f"   Settings: C={c_val}, Gamma={g_val}")

    #use a pipeline to avoid data leakage
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', GPU_SVC(
            kernel='rbf',
            class_weight='balanced',#balances the different number of samples per class
            cache_size=2000,
            probability=False,
            C=c_val,
            gamma=g_val
        ))
    ])

    # Run 5-Fold Cross Validation
    print("    Running 5-Fold CV...", end=" ")
    start_t = time.time()

    cv_scores = cross_val_score(pipe, X, y, cv=cv_strategy, scoring='accuracy')

    end_t = time.time()

    # Mean accuracy
    mean_acc = cv_scores.mean() * 100
    # Standard deviation
    std_acc = cv_scores.std() * 100

    print(f"Finished in {end_t - start_t:.2f} sec")
    print(f"    Mean Accuracy: {mean_acc:.2f}% (±{std_acc:.2f}%)")
    print(f"    Scores per fold: {np.round(cv_scores * 100, 2)}")

    # FINAL TRAINING ON 100% OF THE DATA (FOR SVs)
    print(f"    Training on the full dataset (100%) ")

    # Fit on the whole dataset
    pipe.fit(X, y)

    #  SUPPORT VECTORS
    svm_model = pipe.named_steps['svm']
    try:
        if hasattr(svm_model, 'n_support_') and svm_model.n_support_ is not None:
            total_sv = np.sum(svm_model.n_support_)
            print(f"    Total Support Vectors (Final Model): {int(total_sv)}")
        elif hasattr(svm_model, 'support_') and svm_model.support_ is not None:
             total_sv = len(svm_model.support_)
             print(f"    Total Support Vectors: {total_sv} (Estimated via indices)")
        else:
            print("    Support Vectors info: Not Available")
    except Exception as e:
        print(f"   Could not count the support vectors: {e}")


**4.3 Poly Kernel**

In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import cuml
from cuml.svm import SVC as GPU_SVC
print(f" GPU Mode Active: cuML version {cuml.__version__}")

# =============================================================================
# 1. LOAD THE DATA
# =============================================================================
print("\n--- 1. Loading data ---")

if os.path.exists('datasetTV.csv') and os.path.exists('datasetTest.csv'):
    train_df = pd.read_csv('datasetTV.csv', header=None)
    test_df = pd.read_csv('datasetTest.csv', header=None)
    print(" Files found and loaded!")
else:
    raise SystemExit(" Error: the files are missing.")

# Convert to float32 for better GPU performance
trainData = train_df.values.astype('float32')
testData = test_df.values.astype('float32')

#split into features and labels
X = trainData[:, :-1]
y = trainData[:, -1]

# 80-20 split (the 20 is the validation hold-out)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# COMPUTE THE SCALE VALUE (1 / (n_features * X.var()))
n_features = X_train.shape[1]
gamma_scale_approx = 1.0 / n_features
print(f"Training Set: {X_train.shape}")
print(f"  The 'scale' value is: {gamma_scale_approx:.6f}")

# =============================================================================
# 2. PIPELINE SETUP & GRID SEARCH
# =============================================================================
print("\n--- 2. Starting Grid Search (Polynomial only) ---")

#use a pipeline to avoid data leakage
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', GPU_SVC(
        kernel='poly',
        class_weight='balanced',#balances the different number of samples per class
        cache_size=2000,
        probability=False
    ))
])

#For the other experiments (fine searches) with Poly we just changed the hyperparameter values
param_grid = {#Polynomial Kernel
    'svm__degree': [2, 3, 4],
    'svm__C': [1, 5, 10],
    'svm__gamma': ['scale', 0.01],
    'svm__coef0': [0.0, 1.0, 2.0]
}

# Run GridSearch with 5-Fold Cross Validation.
grid_search = GridSearchCV(
    pipe,  #use the pipeline we built
    param_grid,
    refit=True,  #the model is retrained on all the data (80%)
    verbose=2,
    cv=5,
    n_jobs=1,
    return_train_score=True
)

start_time = time.time()
grid_search.fit(X_train, y_train)
end_time = time.time()
print(f" Finished in {end_time - start_time:.2f} seconds.")

# =============================================================================
# 3. ANALYSIS AND VISUALIZATION OF THE RESULTS
# =============================================================================
results_df = pd.DataFrame(grid_search.cv_results_)

# Get the parameters to make readable labels
results_df['param_degree'] = results_df['param_svm__degree']
results_df['param_C'] = results_df['param_svm__C']
results_df['param_gamma'] = results_df['param_svm__gamma']
results_df['param_coef0'] = results_df['param_svm__coef0']

# Labeling function so the parameters show in every case
def get_full_label(row):
    d = row['param_degree']
    c = row['param_C']
    g = row['param_gamma']
    coef = row['param_coef0']

    g_str = f"scale" if g == 'scale' else f"{g}"

    return f"Deg:{d}, C:{c}, G:{g_str}, Coef:{coef}"

results_df['params_str'] = results_df.apply(get_full_label, axis=1)

# Sort the results by validation performance
results_df_sorted = results_df.sort_values(by='mean_test_score', ascending=False).reset_index(drop=True)

print("\n" + "="*90)
print("SORTED RESULTS (POLYNOMIAL KERNEL)")
print("-" * 90)
print(f"{'Parameters':<50} | {'Train':<8} | {'Val(CV)':<8}")
print("-" * 90)
# Print a table to compare train vs validation score (overfitting check)
for index, row in results_df_sorted.iterrows():
    train_acc = row.get('mean_train_score', 0) * 100
    val_acc = row['mean_test_score'] * 100
    print(f"{row['params_str']:<50} | {train_acc:.2f}%   | {val_acc:.2f}%")
print("="*90 + "\n")

# --- PLOTS IN GROUPS OF 8 SO THEY STAY READABLE ---
BATCH_SIZE = 8
total_experiments = len(results_df_sorted)

for i in range(0, total_experiments, BATCH_SIZE):
    subset_df = results_df_sorted.iloc[i : i + BATCH_SIZE]
        # Prepare the data for seaborn
    acc_data_subset = subset_df[['params_str', 'mean_train_score', 'mean_test_score']].melt(
        id_vars='params_str', var_name='Type', value_name='Accuracy'
    )
    acc_data_subset['Type'] = acc_data_subset['Type'].replace({
        'mean_train_score': 'Train Accuracy', 'mean_test_score': 'Validation Accuracy'
    })

    fig, ax = plt.subplots(1, 2, figsize=(22, 9))
    plt.suptitle(f"Results: experiments {i+1} to {min(i+BATCH_SIZE, total_experiments)} (Poly)", fontsize=16)

    # Plot Accuracy
    sns.barplot(x='params_str', y='Accuracy', hue='Type', data=acc_data_subset, ax=ax[0], palette='viridis')
    ax[0].set_title('Accuracy Comparison', fontsize=15, fontweight='bold')
    ax[0].set_ylim(0.60, 1.01)
    ax[0].grid(axis='y', linestyle='--', alpha=0.5)
    ax[0].set_xticklabels(ax[0].get_xticklabels(), rotation=45, ha='right', fontsize=9)

    # Plot Time
    sns.barplot(x='params_str', y='mean_fit_time', data=subset_df, ax=ax[1], palette='Reds_r')
    ax[1].set_title('Training Time (Sec)', fontsize=15, fontweight='bold')
    ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=45, ha='right', fontsize=9)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# =============================================================================
# 4. EVALUATION OF THE BEST MODEL
# =============================================================================
best_model = grid_search.best_estimator_
y_pred_val = best_model.predict(X_val)

if hasattr(y_pred_val, 'get'): y_pred_val = y_pred_val.get()
if hasattr(y_pred_val, 'to_numpy'): y_pred_val = y_pred_val.to_numpy()

print(f"\n Best Params: {grid_search.best_params_}")
print("--- Classification Report (Validation Set) ---")
print(classification_report(y_val, y_pred_val, digits=4))

# Confusion matrix to show the classification errors
cm = confusion_matrix(y_val, y_pred_val)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges')
plt.title('Confusion Matrix (Polynomial Kernel)')
plt.show()



**4.3.1 Evaluation of the best models (MEAN Acc and Mean SVs)**

In [ ]:
import pandas as pd
import numpy as np
import time
import os
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import cuml
from cuml.svm import SVC as GPU_SVC
print(f" GPU Mode Active: cuML version {cuml.__version__}")

# =============================================================================
# 1. LOAD THE DATA
# =============================================================================
print("\n--- 1. Loading data ---")

if os.path.exists('datasetTV.csv') and os.path.exists('datasetTest.csv'):
    train_df = pd.read_csv('datasetTV.csv', header=None)
    test_df = pd.read_csv('datasetTest.csv', header=None)
    print(" Files found!")
else:
    raise SystemExit(" Error: the files are missing.")

# Convert to float32 for better GPU performance
trainData = train_df.values.astype('float32')
testData = test_df.values.astype('float32')

#split into features and labels
X = trainData[:, :-1]
y = trainData[:, -1]

# =============================================================================
# 2. 5-FOLD CV FOR THE 3 POLY MODELS
# =============================================================================
print("\n" + "="*90)
print("5-FOLD CROSS VALIDATION ANALYSIS (POLY KERNEL - FIXED SV)")
print("="*90)

models_to_test = [
    # 1st experiment
    {'degree': 3, 'C': 6.0, 'gamma': 0.009, 'coef0': 0.3, 'name': 'Poly_1_(D3_C6_G0.009_Coef0.3)'},

    # 2nd experiment
    {'degree': 3, 'C': 1.0, 'gamma': 0.009, 'coef0': 0.5, 'name': 'Poly_2_(D3_C1_G0.009_Coef0.5)'},

    # 3rd experiment
    {'degree': 3, 'C': 5.0, 'gamma': 0.01,  'coef0': 0.5, 'name': 'Poly_3_(D3_C5_G0.01_Coef0.5)'}
]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for params in models_to_test:
    d_val = params['degree']
    c_val = params['C']
    g_val = params['gamma']
    coef_val = params['coef0']
    name = params['name']

    print(f"\n CHECK: {name}")
    print(f"   Settings: Deg={d_val}, C={c_val}, Gamma={g_val}, Coef0={coef_val}")

    fold_accuracies = []
    fold_svs = []

    start_cv_time = time.time()

    for fold_idx, (train_index, val_index) in enumerate(skf.split(X, y)):
        X_train_fold, X_val_fold = X[train_index], X[val_index]
        y_train_fold, y_val_fold = y[train_index], y[val_index]

        #use a pipeline to avoid data leakage
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('svm', GPU_SVC(
                kernel='poly',
                degree=d_val,
                gamma=g_val,
                coef0=coef_val,
                C=c_val,
                class_weight='balanced',#balances the different number of samples per class
                cache_size=2000,
                probability=False
            ))
        ])

        pipe.fit(X_train_fold, y_train_fold)

        # Score
        score = pipe.score(X_val_fold, y_val_fold) * 100
        fold_accuracies.append(score)

        # --- COUNT THE SUPPORT VECTORS
        svm_model = pipe.named_steps['svm']
        try:
            # 1. Try to read n_support_
            if hasattr(svm_model, 'n_support_') and svm_model.n_support_ is not None:
                # Convert to CPU numpy if it is a GPU array
                raw_sv = svm_model.n_support_
                if hasattr(raw_sv, 'to_numpy'):
                    sv_array = raw_sv.to_numpy()
                elif hasattr(raw_sv, 'get'):
                    sv_array = raw_sv.get()
                else:
                    sv_array = np.array(raw_sv)

                sv_count = int(np.sum(sv_array))
                fold_svs.append(sv_count)

            # 2. Alternative: count the indices
            elif hasattr(svm_model, 'support_') and svm_model.support_ is not None:
                sv_count = len(svm_model.support_)
                fold_svs.append(sv_count)
            else:
                fold_svs.append(0) # If everything fails
        except Exception as e:
            # Fallback in case of error
            fold_svs.append(0)

    end_cv_time = time.time()
    # Mean accuracy
    mean_acc = np.mean(fold_accuracies)
    # Mean number of SVs
    mean_sv = np.mean(fold_svs)

    print(f"     CV time: {end_cv_time - start_cv_time:.2f} sec")
    print(f"    MEAN Accuracy (5-Fold): {mean_acc:.4f}%")
    print(f"    MEAN Support Vectors:    {int(mean_sv)}")
    print(f"   (Scores: {[f'{x:.2f}%' for x in fold_accuracies]})")

**4.4.1 Poly_1 model with Bootstrapping and Feature Bagging**

In [ ]:
import pandas as pd
import numpy as np
import time
import os
import gc
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import BaggingClassifier

import cuml
from cuml.svm import SVC as GPU_SVC

print(f" GPU Mode Active: cuML version {cuml.__version__}")

# =============================================================================
# 1. LOAD THE DATA
# =============================================================================
# Check that the files exist
if os.path.exists('datasetTV.csv') and os.path.exists('datasetTest.csv'):
    train_df = pd.read_csv('datasetTV.csv', header=None)
    test_df = pd.read_csv('datasetTest.csv', header=None)
else:
    raise SystemExit(" Error: the files are missing.")

# Convert to float64 for numerical precision
trainData = train_df.values.astype('float64')
testData = test_df.values.astype('float64')

#split into features and labels
X = trainData[:, :-1]
y = trainData[:, -1]

print(f" Data loaded: Train {X.shape}, Test {testData.shape}")

# =============================================================================
# 2. MODEL DEFINITION
# =============================================================================
def get_model():#Returns the Pipeline with the scaler, the SVM and the bagging
    # SVM
    svm = GPU_SVC(
        kernel='poly', degree=3, C=6.0, gamma=0.009, coef0=0.3,
        class_weight='balanced', probability=True, cache_size=500
    )

    # Bagging with 80 estimators to reduce the variance
    bagging = BaggingClassifier(
        estimator=svm,
        n_estimators=80,
        max_samples=0.8, # Each model sees 80% of the data (Bootstrap Sampling)
        max_features=0.7, # Each model sees 70% of the features (Feature Bagging)
        bootstrap=True,
        bootstrap_features=False,
        n_jobs=1,
        random_state=42
    )
    # Pipeline: apply StandardScaler to each data subset
    return Pipeline([
        ('scaler', StandardScaler()),
        ('bagging', bagging)
    ])

# =============================================================================
# 3. FINAL TRAINING (FULL TRAINING) & SAVE
# =============================================================================
print("\n Starting training on the WHOLE dataset...")

# Free memory
gc.collect()

final_pipe = get_model()
start_train = time.time()

try:
    # Fit on 100% of the data
    final_pipe.fit(X, y)
    print(f" Full Training finished in {time.time() - start_train:.2f} sec.")

    # Predict on the test set
    print(" Creating predictions for the Test Set...")
    preds = final_pipe.predict(testData)

    # Save the results
    filename = 'labelsSVM_Bagging_80.npy'
    np.save(filename, preds)
    print(f" Done! Results saved to '{filename}'")

except Exception as e:
    print(f"\n CRITICAL ERROR while running: {e}")


**4.4.2 Poly_2 Pure model**

In [ ]:
import pandas as pd
import numpy as np
import time
import os
import gc
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import cuml
from cuml.svm import SVC as GPU_SVC

print(f" GPU Mode Active: cuML version {cuml.__version__}")

# =============================================================================
# 1. LOAD
# =============================================================================
# Check that the files exist
if os.path.exists('datasetTV.csv') and os.path.exists('datasetTest.csv'):
    train_df = pd.read_csv('datasetTV.csv', header=None)
    test_df = pd.read_csv('datasetTest.csv', header=None)
else:
    raise SystemExit(" Error: the files are missing.")

# Convert to float64 for maximum numerical precision
trainData = train_df.values.astype('float64')
testData = test_df.values.astype('float64')

#split into features and labels
X = trainData[:, :-1]
y = trainData[:, -1]

# =============================================================================
# 2. PURE SVM MODEL DEFINITION
# =============================================================================
def get_model():#Returns the Pipeline with the scaler and the SVM
    svm = GPU_SVC(
        kernel='poly',
        degree=3,
        C=1.0,
        gamma=0.009,
        coef0=0.5,
        class_weight='balanced',#balances the different number of samples per class
        probability=True,
        cache_size=2000
    )

    return Pipeline([
        ('scaler', StandardScaler()),
        ('svm', svm)
    ])

# =============================================================================
# 3. CHECK WITH 5-FOLD CV
# =============================================================================
print("\n STEP 1: Checking Pure SVM accuracy with 5-Fold CV...")
# We use Stratified 5-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []
fold = 1

start_cv = time.time()

for train_index, val_index in skf.split(X, y):
    print(f"    Running Fold {fold}/5...", end=" ")

    # Data split
    X_train_fold, X_val_fold = X[train_index], X[val_index]
    y_train_fold, y_val_fold = y[train_index], y[val_index]

    # New, clean model in every iteration (so nothing carries over from before)
    model = get_model()

    try:
        # Training
        model.fit(X_train_fold, y_train_fold)

        # Evaluation
        fold_preds = model.predict(X_val_fold)
        score = accuracy_score(y_val_fold, fold_preds)
        fold_scores.append(score)
        print(f" Accuracy: {score:.4f}")

    except Exception as e:
        print(f" Error in Fold {fold}: {e}")

    # Clean up, because we load a lot of data on the GPU
    del model, X_train_fold, X_val_fold, y_train_fold, y_val_fold
    try: del fold_preds
    except: pass
    gc.collect()

    fold += 1

print(f"\n CV finished in {time.time() - start_cv:.2f} sec.")
print(f" PURE SVM MEAN ACCURACY: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

# =============================================================================
# 4. FINAL TRAINING (FULL TRAINING)& SAVE
# =============================================================================
print("\n STEP 2: Training Pure SVM on the WHOLE dataset...")

# Free memory
gc.collect()

final_pipe = get_model()
start_train = time.time()

try:
    #Train the model on 100% of the data
    final_pipe.fit(X, y)
    print(f" Full Training finished in {time.time() - start_train:.2f} sec.")

    # Prediction
    print(" Creating predictions...")
    preds = final_pipe.predict(testData)

    # Save
    filename = 'labelsSVM_Pure_Final.npy'
    np.save(filename, preds)
    print(f" Done! Saved as '{filename}'")

except Exception as e:
    print(f"\n CRITICAL ERROR in Full Training: {e}")

**4.4.3 Poly_2 model with Bootstrapping and Feature Bagging**

In [ ]:
import pandas as pd
import numpy as np
import time
import os
import gc
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import BaggingClassifier

import cuml
from cuml.svm import SVC as GPU_SVC

print(f" GPU Mode Active: cuML version {cuml.__version__}")

# =============================================================================
# 1. LOAD THE DATA
# =============================================================================
# Check that the files exist
if os.path.exists('datasetTV.csv') and os.path.exists('datasetTest.csv'):
    train_df = pd.read_csv('datasetTV.csv', header=None)
    test_df = pd.read_csv('datasetTest.csv', header=None)
else:
    raise SystemExit(" Error: the files are missing.")

# Convert to float64 for numerical precision
trainData = train_df.values.astype('float64')
testData = test_df.values.astype('float64')

#split into features and labels
X = trainData[:, :-1]
y = trainData[:, -1]

# =============================================================================
# 2. MODEL DEFINITION
# =============================================================================
def get_model():#Returns the Pipeline with the scaler, the SVM and the bagging
    # SVM
    svm = GPU_SVC(
        kernel='poly', degree=3, C=1.0, gamma=0.009, coef0=0.5,
        class_weight='balanced', probability=True, cache_size=500
    )
    # Bagging with 80 estimators to reduce the variance
    bagging = BaggingClassifier(
        estimator=svm,
        n_estimators=80,
        max_samples=0.8,  # Each model sees 80% of the data (Bootstrap Sampling)
        max_features=0.6,  # Each model sees 60% of the features (Feature Bagging)
        bootstrap=True,
        bootstrap_features=False,
        n_jobs=1,
        random_state=42,
        verbose=10
    )
    return Pipeline([# Pipeline: apply StandardScaler to each data subset
        ('scaler', StandardScaler()),
        ('bagging', bagging)
    ])

# =============================================================================
# 3. CHECK WITH A SIMPLE SPLIT (80% Train - 20% Val), NOT 5 FOLD, SO IT RUNS FASTER
# =============================================================================
print("\n STEP 1: Accuracy check (Validation Split)...")

# Split
X_train_s, X_val_s, y_train_s, y_val_s = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Model for the validation
val_model = get_model()

try:
    start_val = time.time()
    val_model.fit(X_train_s, y_train_s)

    # Predict on the validation set
    val_preds = val_model.predict(X_val_s)
    val_acc = accuracy_score(y_val_s, val_preds)

    print(f" Validation finished in {time.time() - start_val:.2f} sec.")
    print(f" VALIDATION ACCURACY: {val_acc:.4f}")
    print("\nClassification Report:\n")
    print(classification_report(y_val_s, val_preds))

except Exception as e:
    print(f" Error in Validation: {e}")

# FREE MEMORY (VERY IMPORTANT)
print("\n Freeing GPU memory...")
del val_model, X_train_s, X_val_s, y_train_s, y_val_s
try:
    del val_preds
except: pass
gc.collect()

# =============================================================================
# 4. FINAL TRAINING (FULL TRAINING)& SAVE
# =============================================================================
print("\n STEP 2: Training the final model on the WHOLE dataset...")

final_pipe = get_model()
start_train = time.time()

try:
    final_pipe.fit(X, y)
    print(f" Full Training finished in {time.time() - start_train:.2f} sec.")

    # Prediction
    print(" Creating predictions...")
    preds = final_pipe.predict(testData)
    np.save('labelsSVM_Bagging_Final_SplitChecked_new.npy', preds)
    print(" Done! Saved as 'labelsSVM_Bagging_Final_SplitChecked.npy'")

except Exception as e:
    print(f"\n CRITICAL ERROR in Full Training: {e}")

**5 Ensemble Learning ( Majority Voting)**

In [6]:
import numpy as np
import os
from scipy import stats

# =============================================================================
# 1. FILES & NAMES
# =============================================================================
file_poly1 = 'labelsSVM_Bagging_80.npy'           # Poly 1 with Boosting
file_poly2 = 'labelsSVM_Pure_Final (1).npy'       # Poly 2
file_poly3 = 'labelsSVM_Bagging_Final_5FoldChecked (4).npy' # Poly 3, which is really Poly_2 with boosting

print(f" Loading files:\n  Poly 1: {file_poly1}\n  Poly 2: {file_poly2}\n  Poly 3: {file_poly3}\n")

# =============================================================================
# 2. LOAD & CHECK
# =============================================================================
if os.path.exists(file_poly1) and os.path.exists(file_poly2) and os.path.exists(file_poly3):

    # Load the data
    p1 = np.load(file_poly1)
    p2 = np.load(file_poly2)
    p3 = np.load(file_poly3)

    # Check they all have the same size
    if p1.shape == p2.shape == p3.shape:
        print(f" Loaded successfully! Total samples: {len(p1)}")

        # =====================================================================
        # 3. THE VOTE (MAJORITY VOTING)
        # =====================================================================
        # Stack the 3 arrays into one array
        all_preds = np.stack((p1, p2, p3), axis=1)

        # Compute the mode (VOTE)
        mode_res = stats.mode(all_preds, axis=1, keepdims=True)

        final_preds = mode_res.mode.flatten()      # The final label (initial)
        agreement_count = mode_res.count.flatten() # How many agreed

        # PRIORITY TO POLY 2 WHEN ALL VOTES DIFFER ---
        # Find the positions where all 3 disagree (count == 1)
        chaos_indices = np.where(agreement_count == 1)[0]

        if len(chaos_indices) > 0:
            print(f"\n Found {len(chaos_indices)} cases of full disagreement (1 vs 1 vs 1).")
            print("   -> Rule applied: Poly 2 is picked as the winner.")

            #Take the value of p2
            final_preds[chaos_indices] = p2[chaos_indices]

        # =====================================================================
        # 4. STATISTICS & ANALYSIS
        # =====================================================================
        total = len(final_preds)
        unanimous = np.sum(agreement_count == 3)   # All 3 agreed
        split_decision = np.sum(agreement_count < 3) # 2 agreed, or none

        print("\n" + "="*60)
        print(" VOTING RESULTS (VOTING REPORT)")
        print("="*60)
        print(f" Unanimous decisions (3/3):       {unanimous}  ({unanimous/total*100:.2f}%)")
        print(f"  Majority decisions:    {split_decision}  ({split_decision/total*100:.2f}%)")
        print("-" * 60)

        # =====================================================================
        # 5. PRINT ALL THE DISAGREEMENTS
        # =====================================================================
        if split_decision > 0:
            print("\n FULL LIST OF DISAGREEMENTS (ALL CASES):")
            print(f"{'Index':<8} | {'Poly 1':<8} {'Poly 2':<8} {'Poly 3':<8} |  Winner")
            print("-" * 60)

            # Find the rows where the 3 did NOT all agree
            conflict_indices = np.where(agreement_count < 3)[0]

            for i in conflict_indices:
                v1 = p1[i]
                v2 = p2[i]
                v3 = p3[i]
                winner = final_preds[i]


                note = "(Poly2 Rule)" if agreement_count[i] == 1 else ""

                print(f"#{i:<7} | {v1:<8} {v2:<8} {v3:<8} ->  {winner} {note}")

            print("-" * 60)
            print(f"In total {len(conflict_indices)} cases were decided by vote.")

        else:
            print("\n All 3 models agree everywhere!")

        # =====================================================================
        # 6. SAVE THE FINAL FILE
        # =====================================================================
        save_name = 'labels__Ensemble_Learning.npy'
        np.save(save_name, final_preds)

        print("\n" + "="*60)
        print(f"FINAL FILE SAVED: {save_name}")
        print("="*60)

    else:
        print(" ERROR: the files have a different number of samples!")
        print(f"Poly 1: {p1.shape}, Poly 2: {p2.shape}, Poly 3: {p3.shape}")

else:
    print(" FILES MISSING! Check the names and run it again.")

 Loading files:
  Poly 1: labelsSVM_Bagging_80.npy
  Poly 2: labelsSVM_Pure_Final (1).npy
  Poly 3: labelsSVM_Bagging_Final_5FoldChecked (4).npy

 Loaded successfully! Total samples: 6955

 VOTING RESULTS (VOTING REPORT)
 Unanimous decisions (3/3):       6743  (96.95%)
  Majority decisions:    212  (3.05%)
------------------------------------------------------------

 FULL LIST OF DISAGREEMENTS (ALL CASES):
Index    | Poly 1   Poly 2   Poly 3   |  Winner
------------------------------------------------------------
#3       | 5.0      2.0      2.0      ->  2.0 
#17      | 2.0      2.0      5.0      ->  2.0 
#33      | 4.0      3.0      4.0      ->  4.0 
#67      | 3.0      4.0      4.0      ->  4.0 
#115     | 4.0      5.0      4.0      ->  4.0 
#184     | 3.0      3.0      4.0      ->  3.0 
#212     | 2.0      5.0      5.0      ->  5.0 
#233     | 5.0      2.0      5.0      ->  5.0 
#386     | 4.0      5.0      4.0      ->  4.0 
#419     | 2.0      2.0      5.0      ->  2.0 
#464     |